# K Nearest Neighbours

 * Animation of KNN
 * Used to generate images for slides

## Load

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")

Load in a dataset
 * Should have numerical features
 * Labeled (supervised)

The Iris data is included in the `sklearn.datasets` collection, which is convenient.
However, as we will see in later notebooks, `pandas` provides functions to read data from many
data sources, e.g., CSV files.

Extract the relevant data from the iris dict, using standard machine learning terminology like `X` (the features), `y` (the target). The remaining data items are self-explanatory.

In [ ]:
from sklearn import datasets
iris = datasets.load_iris()
dataset_name = "IRIS"
X, y, feature_names, target_names = iris.data, iris.target, iris.feature_names, iris.target_names

Now we add these data components into a pandas dataframe, which is the standard object used to store tabular data for machine learning.

Adding the name of the variable on the last line of the code cell, by itself, causes Jupyter to display that variable for us to check that it is what we expect. This is a good practice to follow more generally while developing notebooks - check as you go!

Note the use of

- a _list comprehension_ to clean up the feature names for use in the dataframe, and
- a DataFrame constructor. Refer to its [documentation here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).

In [ ]:
colNames = [ cn.replace(' (cm)','').replace(' ','_') for cn in feature_names ]
data = pd.DataFrame(data=X, columns=colNames)
data['species'] = y
data

## Splitting the data

The most robust way to assess how well a model is doing is to hold back some of the data and then see how the model succeeds in predicting the target on this held-back data.

Because the held-back data includes the target values for each of its observations, we can then compare the __predicted__ target values with the __actual__ target value provided in the data.

So, after the data has been loaded into the notebook, the next step is to split it into a _training_ set and a _test_ set. During the __training phase__, the model sees the feature and target values of the _training_ set, and during the subsequent __validation phase__, the trained model is given the feature values of the _test_ set and asked to predict the corresponding target values of the test set, which are compared with the __known__ actual target values.

In the code below, we choose an 80:20 (training:test) split of the data, which is stratified so that the training set has the same distribution of rows by target value as the full dataset. It is easy to tell that the full data set has 50 rows of each iris species. The resulting training set keeps `0.8 * 50 = 40` rows of each of the 3 species, leaving `0.2 * 50 = 10` rows of each species in the held-out (test) set.

In [ ]:
from sklearn.model_selection import train_test_split
df_y = data['species']
# Get a list of the feature names - just get the column names that do not match the
# name of the target column
featureNames = [ j for j in list(data.columns) if j != 'species' ]
df_X = data[featureNames]
# By setting the random_state (randoim number generator seed) to a constant, this
# code will give the same result each time it is run, even though the split is random
X_train, X_test, y_train, y_test = train_test_split(df_X, df_y, test_size=0.2, random_state=42, stratify=y)
# For the plots later, it is often more convenient to combine the training X and y into a single dataframe
data_train = pd.concat([X_train, y_train], axis=1)
# Note that the training set has 120 of the original rows in the data, and the row indexes are those
# of the original dataframe, and are in random order becauzse of the random split
data_train

## Exploratory Data Analysis

In the next few weeks, we will develop a rigorous approach that is designed to prepare data for analysis, and to learn about its behaviour, so that we can choose which types of models might be most suitable.

But first, we should recode the target column to make the species assignment easier to understand.

In [ ]:
n = len(target_names)
fromK = range(n)
toV = target_names

mapping = dict(zip(fromK, toV))
data_train['species'] = data_train['species'].replace(mapping)
data_train

For now, we will just do a few simple visualisations of the data, particularly how it is distributed.

The first [such plot]( shows that the training set is perfectly balanced: 40 of each species, 80% of the original set of rows.

In [ ]:
sns.countplot(data_train, x='species')

Now we are going to "normalize" the dataframe, essentially mapping the feature columns to rows. pandas offers a `melt` function for this purpose. Since there are 4 features, instead of having 120 rows with 4 feature columns, melting the dataframe like this gives 120 x 4 rows with the same data.

In [ ]:
data_train_long = pd.melt(data_train, id_vars = 'species', var_name = 'feature', value_name = 'size')
data_train_long

In [ ]:
ax = sns.stripplot(x = 'feature', y = 'size', hue = 'species', data = data_train_long)
# Reduce the size of the x-tick labels, so they fit better
ax.tick_params(axis="x", labelsize=12)
# Now show the resulting plots.
plt.show()

## Question for you....

_Which feature(s) might be more useful for predicting the species, by classifying observations of those features?_

## Selecting 2 features (for modelling and visualisation)

KNN works in arbitrary dimensions, but to visualise it we need to drop down to two. Using Linear Discrimant Analysis (LDA) we can find the best two dimensions to separate the classes, i.e., the best direction to view the data so that the classes are separated as much as possible.

### FOR YOU

Based on the stripplot above, is this a good choice? Justify your answer....


In [ ]:
# drop down to 2D so we can visualise KNN
if False:
    # Because of the if statement, we do not use Linear Discriminant Analysis...
    from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

    lda = LinearDiscriminantAnalysis(n_components=2)
    X_train = lda.fit_transform(X_train, y_train)
    axis_labels = None
else:
    # ...Instead we just pick two columns
    axis_labels = ["sepal_length", "sepal_width"]
    X_train = X_train[axis_labels]

## Question for YOU

_Lookup up discriminant analysis online. How is it similiar to and how is it different to classification?_

Scaling features is necessary before KNN to avoid undue influence by the features that take the largest values, especially if those values are much larger than the values taken by other features. We prefer to give every feature a chance to explain the data!

Also notice that we switch back to the full dataset here. This is __NOT RECOMMENDED IN PRACTICE__, but a) we are not going to validate our results in this notebook until we have met this aspect in class, and b) the notation simplifies from `X_train` to `X`, etc.

In [ ]:
# scale features (to mean=0, std=1)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

randomize the order observation order - for later visualisation

In [ ]:
np.random.seed(42)
idx = np.random.permutation(range(len(y)))
X, y = X[idx], y[idx]

The following provides an interactive means of trying out various settings,
using GUI controls like sliders, etc.

In [ ]:
import ipywidgets as widgets
import matplotlib.patches as mpatches
from collections import Counter

n = X.shape[0]
n_classes = len(target_names)

markers = ['^', 's', '*']
colors =   ['#377eb8', '#4daf4a', '#a65628', '#984ea3','#999999', '#e41a1c', '#dede00']

m = widgets.IntSlider(value=10, min=1, max=n-1, step=1)
k = widgets.IntSlider(value=1, min=1, max=20, step=1)
summary = widgets.Textarea(value='', layout={'height':'150px', 'width': 'auto'}, disabled=False)
ui = widgets.VBox([
    widgets.HBox([
        widgets.VBox([widgets.Label('Number of observations in train set: $m$'), m]),
        widgets.VBox([widgets.Label('Number of neighbours: $k$'), k])
    ]),
    summary])

Prepare a utility function `f` to draw a typical setup, settings controlled by input parameters

In [ ]:
def f(m,k, show_fig=True, save_fig=False, show_region=True, size=100):
    
    global summary
    
    message = ""
    if k>m:
        message += f"Reducing k to {m}, since number of neighbours cannot be bigger than number of observation in train set.\n"
        k = m
    
    # train set consists of first m observations
    X_train, y_train = X[:m], y[:m]

    # new observation is (m+1)(th) observation
    x_new =  X[m]

    # compute distances from new point to all others in train
    distances = np.linalg.norm(X_train - x_new,axis=1)
    neighbours = sorted(zip(distances,y_train), key=lambda x: x[0])

    # get min radius to reach k nearest neighbours 
    radius = neighbours[k-1][0]

    message +=  f"Location of new observation is ({x_new[0]:.2f}, {x_new[1]:.2f})\n" \
        f"Radius needed to reach {k} nearest neighbours = {radius:.2f}"
    message += "\nCounting neighbours ..."
    counts = Counter([a[1] for a in neighbours[:k]])
    max_count = max([v for _,v in counts.items()])
    max_class = [c for c,v in counts.items() if v==max_count]
    
    for i in range(n_classes):
         message += f"\n\t{target_names[i]:20s} {counts[i]:3d} {'MAX' if i in max_class else ''}"
    if len(max_class)==1:
        message += f"\n{k} nearest neighours suggests that new observation should be in class '{target_names[max_class[0]]}'."
    else:
        message += f"\nHave tie in {k} nearest neighours, reduce k by one and rerun."
    # comment on unsafe k
    if k % n_classes==0:
        message += f"\nNote: Number of neighbours k={k} should not be a multiple of number of classes = {n_classes}."
    
    if show_fig or save_fig:
        fig, ax = plt.subplots(1,1,figsize=(8,8))
    
        # add a circle
        if show_region:
            circle = plt.Circle(x_new, radius, color='r', alpha=0.2)
            ax.add_patch(circle)
        
        for i in range(n_classes):
            plt.scatter(X_train[y_train==i, 0], X_train[y_train==i, 1], s=size, marker=markers[i], alpha=.8, color=colors[i],  label=target_names[i])
        plt.scatter(x_new[0], x_new[1], marker='$?$', s=size, alpha=.8, color='red')

        plt.legend(loc='lower left', shadow=False, scatterpoints=1, frameon=True)
        #plt.axis('equal')
        plt.ylim(-2.6,2.6)
        plt.xlim(-2.6,2.6)
        if axis_labels is not None:
            plt.xlabel(axis_labels[0])
            plt.ylabel(axis_labels[1])
        plt.title(f'{dataset_name} dataset (m={m}, k={k})');

    summary.value = message
        
    if save_fig:
        filename = f"output/knn_{dataset_name}_{m}_{k}_{int(show_region)}.pdf"
        plt.savefig(filename, bbox_inches="tight")
    if not show_fig: plt.close()

    return {"X_train":X_train, "y_train":y_train, "x_new":x_new, "distances": distances, "counts": counts}

Ensure there is an output/ folder in which to save plots our plots.

In [ ]:
import os
# does output/ exist, and is it a directory? 
if not os.path.isdir("output/"):
  # Make the directory, and its parents if necessary, so the full path is valid
  os.makedirs("output/")

Now apply the interactive widget to f (to choose its input argument values)

In [ ]:
out = widgets.interactive_output(f, {'m': m, 'k': k})
display(ui, out)

## Visualising KNN in action

In [ ]:
f(26,1, show_fig=1, save_fig=1, show_region=0, size=200);
f(26,1, show_fig=0, save_fig=1, show_region=0, size=200);
f(26,1, show_fig=0, save_fig=1, show_region=1, size=200);
f(26,3, show_fig=0, save_fig=1, show_region=1, size=200);
f(26,5, show_fig=0, save_fig=1, show_region=1, size=200);
f(26,7, show_fig=0, save_fig=1, show_region=1, size=200);
f(26,9, show_fig=0, save_fig=1, show_region=1, size=200);

## Decision Boundaries

In [ ]:
# First import KNN and the iris data (which is one of the standard datasets provided with sklearn) and prepare it for analysis

In [ ]:
from sklearn import datasets
from sklearn.neighbors import KNeighborsClassifier

iris = datasets.load_iris()
dataset_name = "IRIS"
X, y, target_names = iris.data[:, :2], iris.target, iris.target_names

Choose paler colours (`cmap_light`) for the regions, and strong colours (`cmap_bold`) for the data itself.

In [ ]:
from matplotlib.colors import ListedColormap

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])

The `decision_boundaries` function gathers together the steps needed to display the decision boundaries on a 100x100 mesh grid 

In [ ]:
def decision_boundaries(k, show_fig=True):

#
# Create the nearest neighbors classifier object for a specific k number of neighbors
# Then apply it to the data X (sepal length and width) and y (iris labels)
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X, y)

#
# Layout a rectangular 100x100 grid that covers the entire range of the training data, leaving a margin of 0.1 on each boundary
    x_min, x_max = X[:, 0].min() - .1, X[:, 0].max() + .1
    y_min, y_max = X[:, 1].min() - .1, X[:, 1].max() + .1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))

#
# Looping over the mesh grid points, apply the knn model to predict the label for each grid point 
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
#
# Now colour each grid point based on its prediced label (0,1,2), which picks out the relevant colour from the cmap_light colour map.
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(8,8))
    plt.pcolormesh(xx, yy, Z, cmap=cmap_light, shading='auto')

#
# Looping over the 3 labels...
#   Overlay a plot of the relevant training set data points, using its own `marker` symbol and colour (from the `cmap_bold` colour map)
#   Add labels and a title
#   Save the figure, using a filename that is generated based on variables like `dataset_name` and `k`
    for i in range(3):
        plt.scatter(X[y==i,0], X[y==i, 1], c=['#FF0000', '#00FF00', '#0000FF'][i], marker=markers[i])
        plt.xlabel('sepal length (cm)')
        plt.ylabel('sepal width (cm)')
        plt.title(f"{dataset_name} k-NN decision boundaries ($k={k}$)")
        plt.axis('tight')
        plt.savefig(f"output/knn_{dataset_name}_decision_boundary_{k}.pdf")
# The following code is commented out because it does not seem to work with the notebook_exporter
#        if show_fig:
#            plt.show()
# Actually we don't need this anyway, so we can comment it out
#        plt.show()
    plt.close()

In [ ]:
for k in [1,3,5,7,17]:
    decision_boundaries(k,show_fig=False)

## Question for YOU

What happens to the decision boundaries as `k` increases? How might you find an optimal value of `k`?

`k` is not considered a direct part of the model, like the features. Instead it is considered a hyperparameter, and choosing the best value of k is an example of _hyperparameter tuning_